# Workshop 5. A practice case on a customer survey (solutions)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture05/workshop05_solutions.ipynb)

This workshop is a practice run for the exam assignment. The
notebook has the same structure as sections 1 to 4 of the assignment: the case, data preparation, exploratory
analysis, and estimation and tests. The questions are numbered like the assignment's, and each has a code cell and an
answer cell. Work in your group if you have one; this is a good occasion to try working together on a notebook.

The data are a second RetailCorp file with the same kinds of problems as the lecture file, on other variables. The
lecture notebook shows every step in code, and the code patterns below cover what is new. Under each question there
are two hints: the first names the step, the second names the functions. Try the question before you open them.
You are welcome to use an LLM, but every question can be solved without one, and you should be able to explain each
decision yourself: at the oral exam you will be asked about the choices your group made.

Start with the questions that are not marked; they cover the whole path from raw file to test. Questions marked
*(if time)* are good practice for the assignment, and you can return to them after the workshop.

**Updated 10 November 2026.** The note on the cleaning log below has been corrected. Its last example was missing
`.sum()`, so it did not give a count, and its examples used columns that are not in this data. The examples now use
the columns of this workshop, and the note adds two points: count the rows before you fix a problem, and empty cells
are counted by `~isin`. The solutions notebook shows how each count in the cleaning log is computed.

## Toolbox

| Function | What it does |
|---|---|
| `df.info()`, `df.describe()`, `df["col"].value_counts(dropna=False)` | Types and non-missing counts; numerical summary; the values in a column, including `NaN` |
| `df.duplicated().sum()`, `df.drop_duplicates()` | Count and remove rows that are exact copies of an earlier row |
| `df["col"].str.strip().str.lower()` | Remove spaces at both ends and use lower case |
| `df["col"].str.replace(old, new, regex=False)` | Replace plain text |
| `df["col"].map(dictionary)` | Translate each value; values not in the dictionary become `NaN` |
| `.astype(float)`, `.astype(int)` | Convert; stops with an error if a value does not fit. `int` cannot hold `NaN` |
| `df["col"].replace(code, np.nan)` | Turn a code for missing into `NaN` |
| `df["col"].mask(condition)` | Set the values where the condition is true to `NaN` |
| `df.isna().sum()`, `df["col"].isna().groupby(df["group"]).mean()` | Missing values per column; share missing in each group |
| `df.groupby("group")["col"].agg(["count", "mean", "median"])` | Summary by group |
| `df[cols].corr()`, `plt.hist(x, bins=20)`, `stats.skew(x)` | Correlations, histogram, skewness |
| `stats.t.ppf`, `stats.t.sf`, `stats.norm.ppf` | Critical values and p-values, as in Lectures 2 to 4 |

## Code patterns

Three patterns for what is new today. They are templates and do not run as they stand.

**A number written the Danish way, to float**

```python
text = df["col"].str.replace(" kr.", "", regex=False)    # 1. remove the unit
text = text.str.replace(".", "", regex=False)             # 2. remove the thousand separator
text = text.str.replace(",", ".", regex=False)            # 3. decimal comma to point
df["col"] = text.astype(float)                            # stops with an error if a value does not fit
print(df["col"].dtype, df["col"].isna().sum())            # check
```

**Labels to one spelling, and yes/no to 1 and 0**

```python
key = df["col"].str.strip().str.lower()
print(key.value_counts())                                 # which spellings are left?
df["col"] = key.map({"yes": 1, "no": 0}).astype(int)      # every spelling must be in the dictionary
```

**The cleaning log**

```python
log = []
log.append(("Problem", rows_affected, "Decision", "Consequence if not handled"))
pd.DataFrame(log, columns=["Problem", "Rows", "Decision", "Consequence if not handled"])
```

The tests in section 4 follow the code in the lecture notebooks for Lectures 2 to 4.

<div style="border: 1px solid #9ca3af; border-radius: 8px; padding: 1rem 1.25rem; margin: 0.5rem 0;">

## A quick note on the cleaning log

At Workshop 5, you were asked to keep a log of the changes made during data cleaning. The log helps you explain what you found, what you decided to do, and why the decision matters.

A useful structure is:

| Problem | Rows affected | Decision | Consequence if not handled |
|---|---:|---|---|
| What was wrong with the data? | How many rows had the problem? | What did you do about it? | What could go wrong if you left it as-is? |

The problem, decision, and consequence depend on the data and your reasoning. For **Rows affected**, count the rows that have the problem. Finding that count is also a useful way to check that you have identified the relevant rows correctly.

Count the rows **before** you fix a problem. After `drop_duplicates()`, or after the labels have been translated, the same count gives 0.

### Example: counting duplicates

For exact duplicate rows, count them before removing them:

```python
n_dup = df.duplicated().sum()
```

`duplicated()` marks repeated rows as `True` (excluding the first occurrence), and `sum()` counts those `True` values. You can then record `n_dup` in the log alongside your description, decision, and consequence. Alternatively, compare the number of rows before and after removing duplicates.

### Counting rows with conditions

Pandas conditions produce a Boolean Series (`True`/`False`). Because `True` counts as 1 and `False` as 0, summing the Series counts matching rows. The examples use the `region` and `channel` columns of this workshop's data.

Use `.isin([...])` to check whether a column's value is one of several values:

```python
n_north = df["region"].isin(["Nordjylland", "Midtjylland"]).sum()
```

This counts rows where `region` is either Nordjylland or Midtjylland, spelled exactly like that.

Use `~` (tilde) to negate a condition. Put the condition in parentheses so the negation happens before `.sum()`:

```python
regions = ["Hovedstaden", "Midtjylland", "Syddanmark", "Sjælland", "Nordjylland"]
n_other = (~df["region"].isin(regions)).sum()
```

This counts rows where `region` is none of the five correct spellings: the rows with a label problem.

Use `|` for **OR**. It combines conditions row by row, so each matching row is counted once, even if it matches both:

```python
n_rows = (
    ~df["region"].isin(regions) |
    ~df["channel"].isin(["App", "Web"])
).sum()
```

This counts rows with a label problem in `region` **or** in `channel`.

Be careful when negating multiple conditions. Take the rows where the channel is App **or** the region is Nordjylland:

```python
df["channel"].isin(["App"]) | df["region"].isin(["Nordjylland"])
```

This expression:

```python
~df["channel"].isin(["App"]) | ~df["region"].isin(["Nordjylland"])
```

means the channel is not App **or** the region is not Nordjylland. It is **not** the opposite of the earlier OR condition. To count rows that satisfy neither condition, negate the combined condition and then sum:

```python
n_neither = (~(
    df["channel"].isin(["App"]) |
    df["region"].isin(["Nordjylland"])
)).sum()
```

This is equivalent to requiring both that the channel is not App **and** that the region is not Nordjylland.

Watch out for empty cells: `isin` gives `False` for an empty cell, so `~df["col"].isin([...])` also counts the rows where the value is missing. If a column has missing values, count them separately with `df["col"].isna().sum()`, and count the wrong labels with `(~df["col"].isin([...]) & df["col"].notna()).sum()`.

With these patterns, you should be able to identify and count the rows affected by each cleaning decision.

</div>

## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
raw = pd.read_csv(url + "customer_survey.csv")
alpha = 0.05
print(f"Rows: {len(raw)}, columns: {raw.shape[1]}")

## 1 The case

In September 2026 RetailCorp invited 1,500 customers to a short survey. They were drawn at random from all customers
who ordered at least once between January and June 2026. 600 answered. IT linked each answer to the customer's orders
and spend in January to June. Management asks three questions:

1. What share of customers would recommend RetailCorp to a friend?
2. Is average satisfaction above the target of 7 on the 0 to 10 scale?
3. Do members of the loyalty club spend more than other customers?

The data description from IT:

| Column | Content |
|---|---|
| `customer_id` | Customer number |
| `age` | Age in years |
| `region` | The customer's region |
| `channel` | Where the customer orders most often: App or Web |
| `member` | Member of the loyalty club |
| `orders_h1` | Number of orders, January to June 2026 |
| `spend_h1` | Total spend in DKK, January to June 2026, as exported from the shop system |
| `satisfaction` | "How satisfied are you with RetailCorp?", 0 to 10. "Don't know" is recorded as −1; empty if the question was skipped |
| `recommend` | "Would you recommend RetailCorp to a friend?" |

**1.1** Restate the three questions in your own words. Which population can the answers describe, and who is not in the
data? What could that mean for the results?

<details>
<summary>Hint 1</summary>

Think of the steps from all customers to the 600 rows: who was invited, and who answered. Newbold 1.1 describes
nonresponse.
</details>

<details>
<summary>Hint 2</summary>

No code is needed. Compare the 1,500 invited with the 600 who answered.
</details>

**Answer.**

1. Would most customers recommend RetailCorp: the share of all customers who would, with a margin of error.
2. Is the average satisfaction of all customers above 7, or could the sample average above 7 be chance?
3. Do loyalty club members spend more on average in January to June than customers outside the club?

The population is RetailCorp's customers who ordered at least once in January to June 2026. Customers who did not
order in that period are not in the data, and of the 1,500 invited only 600 answered (40%). The results describe the
customers who were willing to answer (Newbold 1.1). If satisfied customers answer more often than dissatisfied ones,
the share recommending and the mean satisfaction are too high as a picture of all customers. The data cannot show
this, because we know nothing about the 900 who did not answer.

## 2 Data preparation

Keep the raw data in `raw` and clean a copy, `df`. Follow the sequence from the lecture for every problem: look,
decide, do, check, log.

**2.1** Inspect the data. The cell is given: run it and describe every problem you find. For each problem, say whether
Python would stop with an error or give a wrong number without warning.

In [ ]:
raw.info()
print(raw.describe(), "\n")
for col in ["region", "channel", "member", "recommend"]:
    print(raw[col].value_counts(), "\n")
print("Rows that are exact copies of an earlier row:", raw.duplicated().sum())
print(raw["spend_h1"].head(4).tolist())

**Answer.**

- 12 rows are exact copies of an earlier row (answers submitted twice).
- `region` has 16 spellings for five regions: "Region ...", lower case, English names. `channel` has six spellings for
two channels; `member` and `recommend` have eight each (Yes, yes, Y, Ja, No, no, N, Nej).
- `spend_h1` is text (`object`): Danish format with a thousand separator, a decimal comma and "kr.". It cannot be used
in calculations until it is converted (Python stops with an error).
- `satisfaction` has −1, the code for "don't know", which pulls the mean down without a warning. It also has 82 empty
cells.
- `age` has 12 empty cells, a minimum of 0 and a maximum of 2008: some ages were typed as a birth year. The mean age,
57.3, is wrong without a warning.
- `spend_h1` has a few very large values (seen after conversion in 2.2e).

In [ ]:
df = raw.copy()
log = []

**2.2a** Duplicate rows. Remove them and check that every customer occurs once.

<details>
<summary>Hint 1</summary>

Exact copies of a row are the same answer submitted twice; keep one copy. Check that `customer_id` is unique
afterward.
</details>

<details>
<summary>Hint 2</summary>

`df.duplicated().sum()`, `df.drop_duplicates()`, `df["customer_id"].duplicated().sum()`, `len(df)`.
</details>

In [ ]:
n_dup = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Removed {n_dup} rows. Rows: {len(df)}, customer ids that occur more than once: {df['customer_id'].duplicated().sum()}")
log.append(("Duplicate rows", n_dup, "Removed: the same answer submitted twice",
            "n = 612 instead of 600; twelve customers count twice"))

**Answer.** 12 rows were exact copies, so 600 customers remain, each once. Keeping them would count twelve
customers twice and make every standard error a little too small.

**2.2b** Inconsistent labels. Give `region` and `channel` one spelling per group, and check that no label was missed.

<details>
<summary>Hint 1</summary>

Remove spaces and use lower case first, so that only the real differences remain. Then translate each spelling. A
label you forgot becomes `NaN`, so count missing values afterward.
</details>

<details>
<summary>Hint 2</summary>

`.str.strip()`, `.str.lower()`, `.str.replace("region ", "", regex=False)`, `.map({...})`,
`.value_counts(dropna=False)`. Count the rows affected with `~df["region"].isin([...])`.
</details>

In [ ]:
regions = ["Hovedstaden", "Midtjylland", "Syddanmark", "Sjælland", "Nordjylland"]
n_region = (~df["region"].isin(regions)).sum()
n_channel = (~df["channel"].isin(["App", "Web"])).sum()
n_rows = (~df["region"].isin(regions) | ~df["channel"].isin(["App", "Web"])).sum()

key = df["region"].str.strip().str.lower().str.replace("region ", "", regex=False)
print(key.value_counts())
df["region"] = key.map({"hovedstaden": "Hovedstaden", "capital region": "Hovedstaden",
                        "midtjylland": "Midtjylland", "central denmark": "Midtjylland",
                        "syddanmark": "Syddanmark", "southern denmark": "Syddanmark",
                        "sjælland": "Sjælland", "zealand": "Sjælland",
                        "nordjylland": "Nordjylland", "north denmark": "Nordjylland"})
df["channel"] = df["channel"].str.strip().str.lower().map({"app": "App", "web": "Web", "website": "Web"})
print(df["region"].value_counts(dropna=False), df["channel"].value_counts(dropna=False))
log.append(("Region and channel labels", n_rows, "Mapped to one spelling per group",
            "Customers with other spellings drop out of comparisons by region or channel"))

**Answer.** 80 rows had a non-standard region label and 40 a non-standard channel label. After
mapping there are five regions and two channels and no `NaN`, so every spelling was covered. Without this step a
comparison of App and Web would leave out every customer whose channel is spelled "app", "APP", "web" or "Website".

**2.2c** Numbers and yes/no stored as text. Convert `spend_h1` to `float`, and `member` and `recommend` to `int` with 1 for
yes and 0 for no. Check that nothing became missing.

<details>
<summary>Hint 1</summary>

Spend: remove the unit, then the thousand separator, then turn the decimal comma into a point (in this order). Yes/no:
first one spelling, then 1 and 0.
</details>

<details>
<summary>Hint 2</summary>

The first two code patterns: `.str.replace(..., regex=False)`, `.astype(float)`, `.str.strip().str.lower()`,
`.map({...})`, `.astype(int)`, `.isna().sum()`.
</details>

In [ ]:
print(df["spend_h1"].head(3).tolist())
text = (df["spend_h1"].str.replace(" kr.", "", regex=False)
                      .str.replace(".", "", regex=False)
                      .str.replace(",", ".", regex=False))
df["spend_h1"] = text.astype(float)
print(df["spend_h1"].dtype, "| missing:", df["spend_h1"].isna().sum())

yes_no = {"yes": 1, "y": 1, "ja": 1, "no": 0, "n": 0, "nej": 0}
n_yes_no = (~df["member"].isin(["Yes", "No"]) | ~df["recommend"].isin(["Yes", "No"])).sum()
for col in ["member", "recommend"]:
    key = df[col].str.strip().str.lower()
    print(col, key.value_counts().to_dict())
    df[col] = key.map(yes_no).astype(int)
print(df[["member", "recommend"]].agg(["sum", "mean"]).round(3))
log.append(("Spend stored as Danish text", len(df), "Unit, thousand separator and decimal comma converted",
            "Spend cannot be used in calculations"))
log.append(("Member and recommend as text", n_yes_no, "Mapped Yes, yes, Y, Ja and No, no, N, Nej to 1 and 0",
            "Counting only \"Yes\" undercounts members and customers who recommend"))

**Answer.** All 600 spend values converted, with no missing values. Order matters: if the comma is replaced first,
removing the dots then turns "817,57" into 81757, so every value is 100 times too large, without a warning.
`member` and `recommend` are now 0/1 integers; 68 rows used another spelling than Yes or No in at least one
of the two. The share of members is 0.457 and the share who would recommend 0.587.

**2.2d** Codes and impossible values. Handle the code −1 in `satisfaction` and the impossible ages. What happens to the mean
satisfaction and the mean age?

<details>
<summary>Hint 1</summary>

−1 is the code for "don't know" (see the data description), so it is a missing answer. An age of 0 is impossible, and
four-digit ages are birth years typed in the wrong field. Decide what to do with each and say why.
</details>

<details>
<summary>Hint 2</summary>

`.value_counts(dropna=False).sort_index()`, `.replace(-1, np.nan)`, `.describe()`,
`df.loc[condition, ["customer_id", "age"]]`, `.mask(condition)`.
</details>

In [ ]:
print(df["satisfaction"].value_counts(dropna=False).sort_index())
n_code = (df["satisfaction"] == -1).sum()
sat_mean_code = df["satisfaction"].mean()
df["satisfaction"] = df["satisfaction"].replace(-1, np.nan)
sat_mean_clean = df["satisfaction"].mean()
print(f"Mean satisfaction with -1 kept: {sat_mean_code:.3f}, after: {sat_mean_clean:.3f}")

bad_age = (df["age"] < 18) | (df["age"] > 100)
print(df.loc[bad_age, ["customer_id", "age"]])
n_age = bad_age.sum()
age_mean_bad = df["age"].mean()
df["age"] = df["age"].mask(bad_age)
age_mean_clean = df["age"].mean()
print(f"Mean age before: {age_mean_bad:.2f}, after: {age_mean_clean:.2f}")
log.append(("Satisfaction coded -1 (don't know)", n_code, "Set to missing",
            f"Mean satisfaction {sat_mean_code:.2f} instead of {sat_mean_clean:.2f}"))
log.append(("Impossible ages (0, or a birth year)", n_age, "Set to missing",
            f"Mean age {age_mean_bad:.1f} instead of {age_mean_clean:.1f}"))

**Answer.** 22 answers were −1. Kept as a score, they pull the mean satisfaction down from 7.260
to 6.909, which matters because question 2 compares the mean with 7. They are set to missing.
5 ages are impossible: one 0 and four birth years. Setting them to missing is the safe choice. Converting a
birth year to an age (2026 minus the year) is also defensible if you say so, since the intended age is clear. The
mean age falls from 57.6 to 44.5 years.

**2.2e** Error or real? Look at the largest values of `spend_h1`. Which values are errors, and how can you tell them from
customers who really spend a lot? What happens to the mean and the median?

<details>
<summary>Hint 1</summary>

A large total is plausible for a customer with many orders. Put spend on the same footing for every customer by
dividing by the number of orders, and look for values far from all the others.
</details>

<details>
<summary>Hint 2</summary>

`df["spend_h1"] / df["orders_h1"]`, `.sort_values(ascending=False).head()`, `df.assign(...)`, `.mask(condition)`,
`.mean()`, `.median()`.
</details>

In [ ]:
print(df["spend_h1"].describe())
per_order = df["spend_h1"] / df["orders_h1"]
print(df.assign(per_order=per_order).sort_values("per_order", ascending=False)
        [["customer_id", "orders_h1", "spend_h1", "per_order"]].head(6).round(2))

extra = per_order > 5000
spend_mean_bad, spend_median_bad = df["spend_h1"].mean(), df["spend_h1"].median()
df["spend_h1"] = df["spend_h1"].mask(extra)
spend_mean_clean, spend_median_clean = df["spend_h1"].mean(), df["spend_h1"].median()
print(f"With them:    mean {spend_mean_bad:.2f}, median {spend_median_bad:.2f}")
print(f"Without them: mean {spend_mean_clean:.2f}, median {spend_median_clean:.2f}")
log.append(("Spend with an extra zero", extra.sum(), "Set to missing (above 5,000 DKK per order; all others below 1,500)",
            f"Mean spend {spend_mean_bad:.0f} instead of {spend_mean_clean:.0f} DKK"))

**Answer.** Three customers spend more than 6,000 DKK per order; every other customer spends less than 1,500 DKK per
order. Their totals (19,054 to 33,291 DKK) look like an extra zero. The customer with the largest real
total, 6,528.73 DKK, has 7 orders and a normal amount per order, so that value is kept. The
three errors are set to missing; dividing by 10 would also be defensible if the shop system could confirm it. The mean
falls from 1,423.62 to 1,299.14 DKK, while the median hardly moves (1,078.05
against 1,075.69).

**2.3** Show the cleaning log as a table. Add one sentence below it: which decisions do you expect to matter most for the
three management questions, and why?

<details>
<summary>Hint 1</summary>

One row per problem: the problem, the number of rows affected, the decision, and the consequence if not handled.
</details>

<details>
<summary>Hint 2</summary>

`pd.DataFrame(log, columns=[...])` (third code pattern). `pd.set_option("display.max_colwidth", 120)` shows long text.
</details>

In [ ]:
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(log, columns=["Problem", "Rows", "Decision", "Consequence if not handled"])

**Answer.** The −1 codes matter most for question 2, because they pull the mean satisfaction below the target. The
extra zeros matter most for question 3, because they sit in the spend variable that is compared. The label mapping
matters for question 1, since counting only "Yes" undercounts the customers who would recommend. Question 4.4 checks
these expectations.

**2.4** Missing values. How many are there in each column after cleaning? Is satisfaction missing more often for customers
with one order than for customers with more? Which of the three types of missing data could this be, and how do you
handle the missing values in section 4?

<details>
<summary>Hint 1</summary>

Compare the share missing in two groups. Then compare the satisfaction of those who answered in the two groups: what
does that mean for the mean of all who answered?
</details>

<details>
<summary>Hint 2</summary>

`df.isna().sum()`, `one = df["orders_h1"] == 1`, `df["satisfaction"].isna().groupby(one).mean()`,
`df.groupby(one)["satisfaction"].agg(["mean", "count", "size"])`.
</details>

In [ ]:
print(df.isna().sum(), "\n")
one = df["orders_h1"] == 1
miss_share = df["satisfaction"].isna().groupby(one).mean()
print("Share of satisfaction missing (True = one order):")
print(miss_share.round(3), "\n")
print(df.groupby(one)["satisfaction"].agg(["mean", "count", "size"]).round(2))

**Answer.** After cleaning, `satisfaction` has 104 missing values (82 skipped and 22 "don't
know"), `age` 17 and `spend_h1` 3. Satisfaction is missing for 33% of one-order
customers and 12% of the others, and one-order customers who answered are less satisfied (6.76
against 7.38). Because the missingness depends on the number of orders, which we can see, it is not missing
completely at random; it fits missing at random. It may also be missing not at random, if dissatisfied customers skip
the question more often within each group; the data cannot rule that out. In section 4 the analyses use the
customers with a value, and we report n. The mean satisfaction of those who answered leans toward repeat customers. If
the scores are missing at random, weighting the two groups by their share of all customers gives 7.23 against
7.26, so the lean is small and does not change the answer to question 2. Filling with the mean would
not fix the lean and would make the interval too narrow (Extension 1).

**2.5** *(if time)* Create the variables you need later: `spend_per_order` (spend divided by the number of orders) and `one_order` (1 if
the customer ordered once, 0 otherwise). Check that they look as you expect.

<details>
<summary>Hint 1</summary>

A new column is computed from existing columns, row by row. A true/false condition becomes 1 and 0 with `astype(int)`.
</details>

<details>
<summary>Hint 2</summary>

`df["spend_h1"] / df["orders_h1"]`, `(df["orders_h1"] == 1).astype(int)`, `.describe()`.
</details>

In [ ]:
df["spend_per_order"] = df["spend_h1"] / df["orders_h1"]
df["one_order"] = (df["orders_h1"] == 1).astype(int)
print(df[["spend_per_order", "one_order"]].describe().round(2))

**Answer.** Spend per order lies between 107 and 1,435 DKK, with a median of 475 DKK;
it is missing for the 3 customers whose spend was set to missing. 142 customers (24%)
ordered once.

**2.6** *(if time)* Save the cleaned data so that you can continue in a later session, either in your Google Drive or by downloading it.
Read the saved file back and check that it matches `df`. Why must the notebook you hand in at the exam still start
from the raw file?

<details>
<summary>Hint 1</summary>

Write the file with `index=False`, read it back, and compare. Look at section "Keeping your work in Colab" in the
lecture notebook.
</details>

<details>
<summary>Hint 2</summary>

`drive.mount`, `os.makedirs`, `df.to_csv(..., index=False)`, `pd.read_csv`, `pd.testing.assert_frame_equal`; or
`files.download`. `df` still has the row numbers from before the duplicates were removed, so compare with
`df.reset_index(drop=True)`.
</details>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
folder = "/content/drive/MyDrive/AQM"
os.makedirs(folder, exist_ok=True)
df.to_csv(folder + "/survey_clean.csv", index=False)

check = pd.read_csv(folder + "/survey_clean.csv")
pd.testing.assert_frame_equal(check, df.reset_index(drop=True))   # stops if they differ
print("Saved file matches:", check.shape)

**Answer.** The saved file has 600 rows and 11 columns and matches the cleaned data. `.equals` would
say False here, because `spend_per_order` differs in the 13th decimal after saving; `assert_frame_equal` allows for
that.
Downloading works the same way with `files.download`, and the file is then uploaded again in the next session. The
notebook you hand in must still load the raw file from the course repository and do the cleaning: the examiner
cannot open your Drive or your Downloads folder, and the cleaning decisions you are asked about at the exam must be
in the notebook.

## 3 Exploratory analysis

Get to know the cleaned data before you test. Aim the analysis at the three questions.

**3.1** Make a table of descriptive statistics for the numerical variables. Compare the mean and the median of `spend_h1`.
What does the comparison say about the shape of the distribution?

<details>
<summary>Hint 1</summary>

When the mean is clearly above the median, a few large values pull the mean up: the distribution is skewed to the
right (Newbold 2.1).
</details>

<details>
<summary>Hint 2</summary>

`df[cols].describe()` with `cols = ["age", "orders_h1", "spend_h1", "spend_per_order", "satisfaction"]`.
</details>

In [ ]:
cols = ["age", "orders_h1", "spend_h1", "spend_per_order", "satisfaction"]
df[cols].describe().round(2)

**Answer.** Mean spend is 1,299 DKK and the median 1,076 DKK. The mean is clearly
above the median, so spend is skewed to the right: most customers spend moderately and a few spend a lot. Age is
roughly symmetric (mean 44.5, median 44). Satisfaction has a mean of 7.26
on the 0 to 10 scale, based on 496 customers.

**3.2** Plot the distribution of `spend_h1` and compute its skewness. Describe the shape in words.

<details>
<summary>Hint 1</summary>

A histogram with about 20 bins, with lines for the mean and the median.
</details>

<details>
<summary>Hint 2</summary>

`plt.hist(x, bins=20)`, `plt.axvline(...)`, `stats.skew(x)`; drop missing values first with `.dropna()`.
</details>

In [ ]:
x = df["spend_h1"].dropna()
plt.hist(x, bins=20, color="#31A9C1")
plt.axvline(x.mean(), color="#DF8E2E", label=f"mean {x.mean():.0f}")
plt.axvline(x.median(), color="#211A52", label=f"median {x.median():.0f}")
plt.xlabel("Spend January to June 2026 (DKK)")
plt.legend()
plt.show()
spend_skew = stats.skew(x)
print(f"Skewness: {spend_skew:.2f}")

**Answer.** The histogram has its peak below 1,000 DKK and a long tail to the right, up to about 6,529 DKK.
The skewness is 1.74. With 273 members and 324 other customers in question 3, the averages are
still close to normal (the central limit theorem, Lecture 1), so a t test for the means is reasonable.

**3.3** *(if time)* Compare the groups that matter for the questions: spend by membership, and satisfaction and the share recommending by
membership and by channel. What patterns do you see?

<details>
<summary>Hint 1</summary>

For each group, the number of customers, the mean and, for spend, the median. Remember that a difference in a sample
still has to be tested.
</details>

<details>
<summary>Hint 2</summary>

`df.groupby("member")["spend_h1"].agg(["count", "mean", "median"])`,
`df.groupby("channel")[["satisfaction", "recommend"]].mean()`.
</details>

In [ ]:
print(df.groupby("member")["spend_h1"].agg(["count", "mean", "median", "std"]).round(1), "\n")
print(df.groupby("member")[["orders_h1", "spend_per_order", "satisfaction", "recommend"]].mean().round(3), "\n")
print(df.groupby("channel")[["satisfaction", "recommend"]].mean().round(3))

**Answer.** Members spent more in the sample: mean 1,471 against 1,154 DKK (medians 1,266 and
979). They also placed more orders (2.72 against 2.27), while spend per order is about the same
(533 against 516 DKK). So members seem to spend more because they order more often. Members and app
users are a little more satisfied and recommend a little more often. Question 3 is tested in 4.3; the other
differences are small and not part of the questions.

**3.4** *(if time)* Compute the correlation matrix of `age`, `orders_h1`, `spend_h1`, `satisfaction`, `recommend` and `member`. Which
correlations stand out?

<details>
<summary>Hint 1</summary>

A correlation measures a straight-line relationship between two variables, from −1 to 1 (Newbold 2.4). Look for the
largest values outside the diagonal.
</details>

<details>
<summary>Hint 2</summary>

`df[cols].corr().round(2)`.
</details>

In [ ]:
cols = ["age", "orders_h1", "spend_h1", "satisfaction", "recommend", "member"]
corr = df[cols].corr().round(2)
corr

**Answer.** Spend goes with the number of orders (0.71), which is expected since spend is the total over all
orders. Satisfaction and recommending go together (0.45). Age is nearly unrelated to everything else. The other
correlations are weak (below 0.2).

## 4 Estimation and tests

For each test, write H<sub>0</sub> and H<sub>1</sub>, say whether the test is one-sided or two-sided and why, check
the conditions, compute the result, and give the answer in words for management. Use α = 0.05. Use the cleaned data
and the customers with a value, and report n.

**4.1** Estimate the share of customers who would recommend RetailCorp, with a 95% confidence interval. Check the condition.

<details>
<summary>Hint 1</summary>

The confidence interval for a proportion from Lecture 2: p̂ ± z<sub>α/2</sub> √(p̂(1 − p̂)/n), with the condition
np̂(1 − p̂) > 5.
</details>

<details>
<summary>Hint 2</summary>

`df["recommend"].mean()`, `.count()`, `np.sqrt`, `stats.norm.ppf(1 - alpha / 2)`.
</details>

In [ ]:
x = df["recommend"]
n, p_hat = x.count(), x.mean()
print(f"n = {n}, p_hat = {p_hat:.3f}, condition n p_hat (1 - p_hat) = {n * p_hat * (1 - p_hat):.1f}")
se = np.sqrt(p_hat * (1 - p_hat) / n)
z = stats.norm.ppf(1 - alpha / 2)
ci_low, ci_high = p_hat - z * se, p_hat + z * se
print(f"SE = {se:.4f}, z = {z:.3f}, 95% interval [{ci_low:.3f}, {ci_high:.3f}]")

**Answer.** 58.7% of the 600 customers would recommend RetailCorp. The condition holds
(np̂(1 − p̂) = 145.5 > 5). The 95% interval is [0.547, 0.626]: we are 95% confident that between
54.7% and 62.6% of the customers who would answer such a survey would recommend RetailCorp. Since only
40% of those invited answered, the share among all customers may differ (question 1.1).

**4.2** Is average satisfaction above the target of 7?

<details>
<summary>Hint 1</summary>

The one-sample t test from Lecture 3, one-sided: H<sub>0</sub>: μ ≤ 7 and H<sub>1</sub>: μ > 7. Use only the
customers with a satisfaction score.
</details>

<details>
<summary>Hint 2</summary>

`.dropna()`, `len`, `.mean()`, `.std(ddof=1)`, `np.sqrt`, `stats.t.ppf(1 - alpha, df=n - 1)`, `stats.t.sf`.
</details>

In [ ]:
s = df["satisfaction"].dropna()
n_s = len(s)
t_sat = (s.mean() - 7) / (s.std(ddof=1) / np.sqrt(n_s))
crit_sat = stats.t.ppf(1 - alpha, df=n_s - 1)
p_sat = stats.t.sf(t_sat, df=n_s - 1)
print(f"n = {n_s}, mean = {s.mean():.3f}, s = {s.std(ddof=1):.3f}")
print(f"t = {t_sat:.3f}, crit = {crit_sat:.3f}, p = {p_sat:.4f}")

**Answer.** H<sub>0</sub>: μ ≤ 7, H<sub>1</sub>: μ > 7, one-sided because management asks whether the target is
exceeded. With 496 answers, the mean is 7.260, and t = 3.385 > 1.648, p =
0.0004: reject H<sub>0</sub>. For management: among the customers who answered, average satisfaction is above
the target of 7 (7.26 on the 0 to 10 scale). A mean this far above 7 would be unlikely if the true
mean were 7.
Satisfaction is missing more often for one-order customers, who are less satisfied, and many invited customers did not
answer, so the average for all customers may be lower.

**4.3** Do loyalty club members spend more on average than other customers?

<details>
<summary>Hint 1</summary>

Two independent groups: the t test for two means from Lecture 4, one-sided, H<sub>1</sub>: μ<sub>member</sub> −
μ<sub>other</sub> > 0. Use Welch's version, since the spreads differ.
</details>

<details>
<summary>Hint 2</summary>

`df.loc[df["member"] == 1, "spend_h1"].dropna()`, `.var(ddof=1)`, `np.sqrt`, the Welch degrees of freedom, `stats.t.sf`.
Check with `stats.ttest_ind(x1, x2, equal_var=False, alternative="greater")`.
</details>

In [ ]:
x1 = df.loc[df["member"] == 1, "spend_h1"].dropna()
x2 = df.loc[df["member"] == 0, "spend_h1"].dropna()
n1, n2 = len(x1), len(x2)
v1, v2 = x1.var(ddof=1) / n1, x2.var(ddof=1) / n2
t_sp = (x1.mean() - x2.mean()) / np.sqrt(v1 + v2)
df_sp = (v1 + v2)**2 / (v1**2 / (n1 - 1) + v2**2 / (n2 - 1))
p_sp = stats.t.sf(t_sp, df=df_sp)
gap_sp = x1.mean() - x2.mean()
print(f"members n = {n1}, mean {x1.mean():.2f}; others n = {n2}, mean {x2.mean():.2f}; gap {gap_sp:.2f} DKK")
print(f"t = {t_sp:.3f}, df = {df_sp:.2f}, p = {p_sp:.6f}")
print(stats.ttest_ind(x1, x2, equal_var=False, alternative="greater"))

**Answer.** H<sub>0</sub>: μ<sub>member</sub> − μ<sub>other</sub> ≤ 0, H<sub>1</sub>: μ<sub>member</sub> −
μ<sub>other</sub> > 0, one-sided because management asks whether members spend more. Members spent 317 DKK
more on average in January to June (273 members, 324 others). Welch's t = 4.157 with 506.6 degrees of
freedom, p < 0.001: reject H<sub>0</sub>. Spend is skewed, but both groups are large, so the averages are close
to normal. For management: members spend more on average than other customers. The customers chose themselves
whether to join the club, so the test shows a difference; it cannot show that membership causes it. 3.3 suggests that
members order more often while spending about the same per order.

**4.4** *(if time)* What did the cleaning change? Run 4.2 again with the −1 codes kept as scores, and 4.3 again with the three extra
zeros kept. Which cleaning decisions changed a conclusion?

<details>
<summary>Hint 1</summary>

Start again from `raw`: remove the duplicates and redo only the steps you need, leaving out the one you want to look
at. Keep the same index as `df`, so that the member column still matches.
</details>

<details>
<summary>Hint 2</summary>

`raw.drop_duplicates()`, `.dropna()`, the conversion from 2.2c without `.mask(...)`. `df["member"]` has the same index
as `raw.drop_duplicates()`, so `spend_raw[df["member"] == 1]` selects the members.
</details>

In [ ]:
base = raw.drop_duplicates()

s_code = base["satisfaction"].dropna()            # -1 kept as a score
t_code = (s_code.mean() - 7) / (s_code.std(ddof=1) / np.sqrt(len(s_code)))
p_code = stats.t.sf(t_code, df=len(s_code) - 1)
print(f"4.2 with -1 kept: n = {len(s_code)}, mean = {s_code.mean():.3f}, t = {t_code:.3f}, p = {p_code:.4f}")

spend_raw = (base["spend_h1"].str.replace(" kr.", "", regex=False)
                             .str.replace(".", "", regex=False)
                             .str.replace(",", ".", regex=False)).astype(float)
r = stats.ttest_ind(spend_raw[df["member"] == 1], spend_raw[df["member"] == 0],
                    equal_var=False, alternative="greater")
t_zero, p_zero = r.statistic, r.pvalue
print(f"4.3 with the extra zeros kept: t = {t_zero:.3f}, p = {p_zero:.4f}")
print(spend_raw.groupby(df["member"]).agg(["mean", "std"]).round(1))

p_yes_only = (base["recommend"] == "Yes").mean()      # 4.1 counting only "Yes"
print(f"4.1 counting only Yes: {p_yes_only:.3f}")

**Answer.** With the −1 codes kept, the mean satisfaction is 6.909, below 7, so t = −0.874 and p =
0.81: the data would no longer show that satisfaction is above the target. With the three extra zeros kept,
t = 1.287 and p = 0.099: the difference in spend would not be significant at 5%. Two of the three wrong
values belong to non-members, so they raise the non-member mean (from 1,154 to 1,330 DKK) and
shrink the gap, and all three inflate the standard deviations. Both decisions change a conclusion. Counting only
"Yes" in 4.1 would give a share of 0.550 instead of 0.587.

## Extension tasks

For those who have time left. They go beyond the questions of the assignment, but they are good preparation for the
analytical discussion at the oral exam.

### Extension 1. Fill instead of drop

Fill the missing satisfaction scores with the mean and run the test in 4.2 again. What happens to n, the mean, the
standard deviation and t? Why should you not report this version?

<details>
<summary>Hint 1</summary>

Every filled value sits exactly at the mean.
</details>

<details>
<summary>Hint 2</summary>

`df["satisfaction"].fillna(df["satisfaction"].mean())`.
</details>

In [ ]:
f = df["satisfaction"].fillna(df["satisfaction"].mean())
t_fill = (f.mean() - 7) / (f.std(ddof=1) / np.sqrt(len(f)))
print(f"n = {len(f)}, mean = {f.mean():.3f}, s = {f.std(ddof=1):.3f}, t = {t_fill:.3f}")

**Answer.** n rises from 496 to 600, the mean stays at 7.260, the standard deviation falls from
1.711 to 1.556, and t rises from 3.385 to 4.095. The 104 filled values carry no
information, yet they count as observations: the standard error is too small and the test claims more certainty than
the data give. Filling also does not remove the lean toward repeat customers.

### Extension 2. More orders or larger orders?

Do members spend more per order than other customers? Test it with `spend_per_order` and compare with 4.3.

<details>
<summary>Hint 1</summary>

The same Welch test as in 4.3, on another variable.
</details>

<details>
<summary>Hint 2</summary>

`stats.ttest_ind(..., equal_var=False, alternative="greater")`.
</details>

In [ ]:
r = stats.ttest_ind(df.loc[df["member"] == 1, "spend_per_order"].dropna(),
                    df.loc[df["member"] == 0, "spend_per_order"].dropna(), equal_var=False, alternative="greater")
t_spo, p_spo = r.statistic, r.pvalue
print(df.groupby("member")["spend_per_order"].mean().round(2))
print(f"t = {t_spo:.3f}, p = {p_spo:.3f}")

**Answer.** Spend per order is about the same in the two groups (t = 0.866, p = 0.193), so the sample
does not show that members spend more per order. Together with 4.3: members spend more in total because they order
more often. This does not show that the club makes customers order more; customers who order often may be the ones
who join.